# 📘 Semaine 6 — PWM (Pulse Width Modulation)

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 cours + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Expliquer** le principe de la modulation de largeur d'impulsion (PWM).
2. **Calculer** la fréquence et le rapport cyclique (duty cycle) d'un signal PWM.
3. **Configurer** un canal PWM sur TIM1, TIM2 ou TIM3.
4. **Contrôler** une LED en intensité variable (dimming) et un servo moteur.
5. **Choisir** la fréquence PWM adaptée à l'application (LED, moteur, servo, buzzer).

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Cours** | Activités 1 à 6 | 1h30 |
| **B — Atelier** | TP6 : LED dimming + servo moteur | 1h30 |
| **C — Homework** | Exercices 1 à 3 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

---
# 🎓 PARTIE A — COURS INTÉGRÉ (1h30)

## 🔹 Activité 1 — Rappel & mise en contexte (10 min)

### 🔄 Rappel de la semaine 5
- **Timer** : PSC → CNT → ARR → Update.
- **Formule** : `F = F_clk / ((PSC+1) × (ARR+1))`.
- **Interruption** : `HAL_TIM_PeriodElapsedCallback`.
- **APB1 timer ×2** : TIM2/TIM3 reçoivent 72 MHz.

### ✍️ Questions flash (2 min)
1. Quelle est la formule de la fréquence d'un timer ? → ...
2. Que se passe-t-il quand CNT dépasse ARR ? → ...
3. Quelle est la différence entre TIM1 et TIM2 ? → ...
4. Comment appelle-t-on le signal qui alterne ON/OFF rapidement ? → ...

### 🎯 Nouveau problème à résoudre
Comment faire **varier progressivement** la luminosité d'une LED **sans changer la tension** ?

---

## 🔹 Activité 2 — Principe du PWM (20 min)

### 📖 2.1 — Définition

Le **PWM** (*Pulse Width Modulation*) est un signal carré périodique dont on fait varier la **durée de l'état haut** (largeur d'impulsion).

```
Duty = 25 %                Duty = 50 %                Duty = 75 %
                                                      
  ┌──┐        ┌──┐          ┌────┐      ┌────┐          ┌──────┐  ┌──────┐
  │  │        │  │          │    │      │    │          │      │  │      │
──┘  └────────┘  └──       ─┘    └──────┘    └──       ─┘      └──┘      └──
  ◄────────── T ──────────►
```

### 📖 2.2 — Grandeurs caractéristiques

| Grandeur | Symbole | Formule |
|---|---|---|
| **Période** | T | 1 / F_pwm |
| **Fréquence** | F_pwm | 1 / T |
| **Temps haut** | T_on | Duty × T |
| **Temps bas** | T_off | (1 − Duty) × T |
| **Rapport cyclique** | Duty | T_on / T |

### 📖 2.3 — Valeur moyenne

La valeur moyenne du signal (utile pour une LED, un moteur) est :
```
V_moy = Duty × V_high
```

Exemple : PWM 3.3 V à 25 % → V_moy = 0.825 V

### 🐍 Simulation Python — Visualisation PWM (10 min)

Générons et affichons plusieurs signaux PWM avec différents duty cycles.

In [ ]:
# ============================================================
# Génération et visualisation d'un signal PWM (ASCII)
# ============================================================

def generer_pwm(n_points, periode_pts, duty):
    """Retourne un signal PWM sous forme de liste de 0/1."""
    signal = []
    seuil = int(duty * periode_pts)
    for i in range(n_points):
        phase = i % periode_pts
        signal.append(1 if phase < seuil else 0)
    return signal

def afficher_pwm(signal, largeur=60, etiquette=""):
    pas = max(1, len(signal) // largeur)
    ligne = etiquette + " "
    for i in range(0, len(signal), pas):
        bloc = signal[i:i+pas]
        m = sum(bloc) / len(bloc)
        if m > 0.5: ligne += "█"
        elif m > 0.1: ligne += "▒"
        else: ligne += " "
    return ligne

# Paramètres
PERIODE_PTS = 20
N_POINTS    = 200   # 10 périodes visibles

print("📊 Signal PWM à 5 duty cycles différents\n")

for duty in [0.10, 0.25, 0.50, 0.75, 0.90]:
    sig = generer_pwm(N_POINTS, PERIODE_PTS, duty)
    v_moy = duty * 3.3
    print(afficher_pwm(sig, etiquette=f"D={int(duty*100):>3}% ({v_moy:.2f} V) :"))

print("\n📌 À retenir :")
print("  → Même fréquence pour tous les signaux.")
print("  → Seule la largeur d'impulsion change.")
print("  → La valeur moyenne change linéairement avec le duty.")

### 📖 2.4 — Pourquoi PWM plutôt que DAC ?

| Critère | PWM | DAC |
|---|---|---|
| Complexité | Très simple | Nécessite un DAC |
| Rendement | Très élevé (transistors en saturation) | Moyen |
| Résolution | Illimitée (théorique) | Limitée (8/10/12 bits) |
| Filtrage | Nécessite un filtre passe-bas | Direct |
| Coût | Quasi nul (timer existant) | Composant dédié |
| Utilisation | LED, moteurs, servo | Audio, précision |

> 💡 Pour la plupart des charges (LED, moteur), la **valeur moyenne** suffit : le PWM est idéal.

---

## 🔹 Activité 3 — PWM sur STM32 (25 min)

### 📖 3.1 — Chaîne PWM dans un timer

```
   F_clk (72 MHz)
        │
        ▼
   ┌──────────┐
   │   PSC    │  divise → f_cnt
   └────┬─────┘
        │
        ▼
   ┌──────────┐
   │   CNT    │  compte 0 → ARR
   └────┬─────┘
        │
        ▼
   ┌──────────┐
   │  CCR1    │  valeur de comparaison
   └────┬─────┘
        │
        ▼
   ┌──────────┐
   │  Sortie  │  OC1 (Output Compare 1)
   │  OC1REF  │
   └──────────┘
```

### 📖 3.2 — Formules PWM

```
                      F_clk
F_pwm  =  ──────────────────────────
           (PSC + 1) × (ARR + 1)

                    CCR
Duty   =  ──────────────────
           (ARR + 1)

CCR    =  Duty × (ARR + 1)
```

> ⚠️ **Attention** : la plage utile de CCR est **0 à ARR+1**.  
> Pour un duty de 100 %, il faut `CCR > ARR`.

### 📖 3.3 — Modes PWM (CCMR)

| Mode | Comportement |
|---|---|
| **PWM Mode 1** | Sortie active tant que `CNT < CCR` |
| **PWM Mode 2** | Sortie active tant que `CNT ≥ CCR` |

**Alignement :**

| Alignement | Description | Usage |
|---|---|---|
| **Edge-aligned** | Comptage up ou down uniquement | LED, buzzer |
| **Center-aligned** | Up puis down | Moteurs (moins d'harmoniques) |

### 🐍 Comparaison Edge vs Center-aligned (10 min)

In [ ]:
# ============================================================
# Comparaison edge-aligned vs center-aligned
# ============================================================

def pwm_edge_up(n_points, arr, ccr):
    """PWM edge-aligned, comptage UP, mode 1."""
    sig = []
    cnt = 0
    for _ in range(n_points):
        sig.append(1 if cnt < ccr else 0)
        cnt += 1
        if cnt > arr:
            cnt = 0
    return sig

def pwm_center(n_points, arr, ccr):
    """PWM center-aligned, mode 1."""
    sig = []
    cnt = 0
    up = True
    for _ in range(n_points):
        sig.append(1 if cnt < ccr else 0)
        if up:
            cnt += 1
            if cnt >= arr:
                up = False
        else:
            cnt -= 1
            if cnt <= 0:
                up = True
    return sig

def afficher(signal, largeur=60):
    pas = max(1, len(signal) // largeur)
    ligne = ""
    for i in range(0, len(signal), pas):
        bloc = signal[i:i+pas]
        m = sum(bloc) / len(bloc)
        if m > 0.5: ligne += "█"
        elif m > 0.1: ligne += "▒"
        else: ligne += " "
    return ligne

ARR = 20
CCR = 10   # duty 50%
N   = 100

print("Edge-aligned (up) :")
print("|" + afficher(pwm_edge_up(N, ARR, CCR)) + "|")
print("\nCenter-aligned :")
print("|" + afficher(pwm_center(N, ARR, CCR)) + "|")

print("\n📌 Différences :")
print("  → Edge : 1 front montant + 1 front descendant par période.")
print("  → Center : fronts symétriques, fréquence de sortie / 2.")
print("  → Center : moins d'harmoniques → idéal moteurs.")

---

## 🔹 Activité 4 — Canaux PWM disponibles (15 min)

### 📖 4.1 — Mapping timers ↔ broches

| Timer | Canal | Broche par défaut | Remap possible |
|---|---|---|---|
| **TIM1** | CH1 | PA8 | — |
| **TIM1** | CH2 | PA9 | — |
| **TIM1** | CH3 | PA10 | — |
| **TIM1** | CH4 | PA11 | — |
| **TIM2** | CH1 | PA0 | PA15, PA5 |
| **TIM2** | CH2 | PA1 | PB3 |
| **TIM2** | CH3 | PA2 | PB10 |
| **TIM2** | CH4 | PA3 | PB11 |
| **TIM3** | CH1 | PA6 | PB4, PC6 |
| **TIM3** | CH2 | PA7 | PB5, PC7 |
| **TIM3** | CH3 | PB0 | PC8 |
| **TIM3** | CH4 | PB1 | PC9 |

> 💡 **Attention** : PA2/PA3 (TIM2_CH3/CH4) sont aussi utilisés par **USART2**.  
> Choisir les broches selon les besoins (voir S10).

### 📖 4.2 — Broches compatibles avec la Blue Pill

| Broche | Timer associé | Usage typique |
|---|---|---|
| PA0 | TIM2_CH1 | PWM LED / moteur |
| PA1 | TIM2_CH2 | PWM LED / servo |
| PA2 | TIM2_CH3 / USART2_TX | PWM ou UART |
| PA3 | TIM2_CH4 / USART2_RX | PWM ou UART |
| PA6 | TIM3_CH1 | PWM |
| PA7 | TIM3_CH2 | PWM |
| PB0 | TIM3_CH3 | PWM |
| PB1 | TIM3_CH4 | PWM |

---

## 🔹 Activité 5 — Applications du PWM (15 min)

### 📖 5.1 — Choix de la fréquence selon l'application

| Application | Fréquence | Duty typique | Remarque |
|---|---|---|---|
| **LED (dimming)** | 1 – 10 kHz | 0 – 100 % | > 100 Hz pour éviter le scintillement |
| **Servo moteur** | 50 Hz | 5 – 10 % (1–2 ms) | Standard RC |
| **Moteur DC** | 10 – 20 kHz | 0 – 100 % | > 20 kHz pour éviter le sifflement |
| **Buzzer passif** | 2 – 4 kHz | 50 % | Fréquence audible |
| **Chauffage (MOSFET)** | 100 Hz – 1 kHz | 0 – 100 % | Selon thermostat |
| **Charger batterie** | 10 – 100 kHz | Variable | Selon chimie |

### 📖 5.2 — Servo moteur : signal standard

```
Période = 20 ms (50 Hz)

  1.0 ms ──▶ position 0°
  1.5 ms ──▶ position 90°  (centre)
  2.0 ms ──▶ position 180°
```

**Duty correspondant :**
```
Duty = T_on / T = 1.0 ms / 20 ms = 5 %   → 0°
Duty = 1.5 ms / 20 ms = 7.5 %            → 90°
Duty = 2.0 ms / 20 ms = 10 %             → 180°
```

### 📖 5.3 — Dimming LED : loi de perception

L'œil perçoit la luminosité de façon **logarithmique**. Un duty linéaire donne une impression de variation non linéaire.

Pour une variation visuellement linéaire :
```
Duty = (valeur / max)^2.2      (loi gamma)
```

### 🐍 Simulation Python — Servo et LED (10 min)

In [ ]:
# ============================================================
# Calculs pour servo moteur et LED
# ============================================================

def config_servo(f_clk_hz, f_pwm_hz=50, angle=90):
    """
    Calcule PSC, ARR, CCR pour positionner un servo.
    - f_pwm_hz : fréquence du servo (50 Hz typique)
    - angle    : angle cible 0..180°
    """
    # 1) Calcul PSC/ARR pour avoir T = 20 ms
    total = f_clk_hz / f_pwm_hz      # = (PSC+1)*(ARR+1)
    psc_m1 = 72
    arr_m1 = int(total / psc_m1)
    # 2) Calcul largeur d'impulsion en µs pour l'angle
    t_on_us = 1000 + (angle / 180.0) * 1000   # 1000..2000 µs
    # 3) CCR = t_on / T × (ARR+1)
    ccr = int((t_on_us / 1e6) * f_pwm_hz * arr_m1)
    return {
        "PSC": psc_m1 - 1,
        "ARR": arr_m1 - 1,
        "CCR": ccr,
        "duty_%": (t_on_us / 20000.0) * 100,
        "t_on_us": t_on_us,
    }

F_CLK = 72_000_000

print("🎯 Configuration servo moteur (50 Hz)\n")
print(f"{'Angle':<8}{'T_on (µs)':<14}{'Duty (%)':<12}{'CCR (ARR=19999)'}")
print("-" * 55)
for angle in [0, 45, 90, 135, 180]:
    cfg = config_servo(F_CLK, f_pwm_hz=50, angle=angle)
    print(f"{angle:>3}°     {cfg['t_on_us']:<14.0f}{cfg['duty_%']:<12.2f}{cfg['CCR']}")

print("\n📌 Pour un servo à 50 Hz : PSC = 71, ARR = 19999 (72e6 / 72 / 20000 = 50 Hz)\n")

# LED : loi gamma
print("💡 Dimming LED — loi gamma 2.2\n")
print(f"{'Valeur':<10}{'Duty linéaire':<18}{'Duty gamma 2.2'}")
print("-" * 45)
for v in [0, 16, 32, 64, 128, 192, 255]:
    lin = v / 255 * 100
    gam = (v / 255) ** 2.2 * 100
    print(f"{v:<10}{lin:<18.1f}{gam:.1f}%")

---

## 🔹 Activité 6 — Programmation HAL (15 min)

### 📖 6.1 — Configuration CubeMX

1. *Timers → TIM2 → Clock Source = Internal Clock*
2. *Channel1 → PWM Generation CH1*
3. *Parameter Settings* :
   - Prescaler (PSC) : `71`
   - Counter Period (ARR) : `999` → F = 1 kHz
   - PWM Mode : `PWM mode 1`
   - Pulse (CCR1) : `500` → duty 50 %
4. Générer le code

### 📖 6.2 — Code HAL

```c
HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_1);
```

**Modifier le duty cycle dynamiquement :**
```c
__HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_1, nouvelle_valeur_ccr);
```

**Modifier la fréquence :**
```c
__HAL_TIM_SET_AUTORELOAD(&htim2, nouvelle_valeur_arr);
```

In [ ]:
/* ============================================================
   PWM LED — fade in / fade out sur PA0 (TIM2_CH1)
   - Fréquence PWM : 1 kHz
   - PSC = 71, ARR = 999
   - CCR variable : 0..1000
   ============================================================ */

#include "main.h"

TIM_HandleTypeDef htim2;

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_TIM2_Init();

    HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_1);

    while (1)
    {
        // Fade in
        for (uint16_t ccr = 0; ccr <= 1000; ccr += 5)
        {
            __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_1, ccr);
            HAL_Delay(5);
        }
        // Fade out
        for (uint16_t ccr = 1000; ccr > 0; ccr -= 5)
        {
            __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_1, ccr);
            HAL_Delay(5);
        }
    }
}

### 📖 6.3 — Version LL (plus rapide)

```c
LL_TIM_EnableCounter(TIM2);
LL_TIM_CC_EnableChannel(TIM2, LL_TIM_CHANNEL_CH1);
LL_TIM_OC_SetCompareCH1(TIM2, 500);   // CCR1 = 500
```

### 📖 6.4 — Version registres

```c
// Activation horloge TIM2
RCC->APB1ENR |= RCC_APB1ENR_TIM2EN;
RCC->APB2ENR |= RCC_APB2ENR_IOPAEN | RCC_APB2ENR_AFIOEN;

// PA0 en AF push-pull 50 MHz (CRL bits [3:0] = 0b1011)
GPIOA->CRL &= ~(0xF << 0);
GPIOA->CRL |=  (0xB << 0);

// Configuration TIM2
TIM2->PSC  = 71;
TIM2->ARR  = 999;
TIM2->CCR1 = 500;

// PWM mode 1 sur CH1 : CCMR1 = 0b0110_1000
TIM2->CCMR1 = (0x6 << 4) | (1 << 3);
TIM2->CCER |= TIM_CCER_CC1E;   // activer sortie CH1

// Démarrer le timer
TIM2->CR1 |= TIM_CR1_CEN;
TIM2->EGR = TIM_EGR_UG;   // forcer update
```

---

## 🔹 Activité 7 — QCM formatif (10 min)

**1. Le rapport cyclique (duty cycle) est :**  
A. T_off / T  
B. T_on / T  
C. F_pwm / F_clk  
D. ARR / PSC

**2. Pour F_clk = 72 MHz, PSC = 71, ARR = 999, la fréquence PWM est :**  
A. 100 Hz  
B. 1 kHz  
C. 10 kHz  
D. 100 kHz

**3. La valeur du registre CCR détermine :**  
A. La fréquence  
B. Le duty cycle  
C. Le prescaler  
D. Le mode

**4. Un servo moteur standard utilise une fréquence PWM de :**  
A. 50 Hz  
B. 500 Hz  
C. 1 kHz  
D. 10 kHz

**5. Pour éviter un sifflement audible sur un moteur DC, il faut :**  
A. F_pwm < 1 kHz  
B. F_pwm ≈ 50 Hz  
C. F_pwm > 20 kHz  
D. F_pwm = 100 Hz

**6. La fonction HAL qui démarre un canal PWM est :**  
A. `HAL_TIM_Base_Start()`  
B. `HAL_TIM_PWM_Start()`  
C. `HAL_TIM_PWM_Init()`  
D. `HAL_GPIO_WritePin()`

### ✅ Corrigé du QCM formatif

| Q | Réponse | Explication |
|---|---|---|
| 1 | **B — T_on / T** | Rapport du temps haut sur la période |
| 2 | **B — 1 kHz** | 72e6 / (72 × 1000) = 1000 Hz |
| 3 | **B — Duty cycle** | CCR / (ARR+1) |
| 4 | **A — 50 Hz** | Standard RC servo |
| 5 | **C — > 20 kHz** | Au-delà de l'audible humain |
| 6 | **B — `HAL_TIM_PWM_Start()`** | Démarrage du canal |

**Mon score : ___ / 6**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP6 — LED dimming + servo moteur

### 🎯 Objectif
Contrôler une LED en intensité variable (fade in/out) sur PA0 (TIM2_CH1) et un servo moteur sur PA1 (TIM2_CH2).

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Configurer TIM2_CH1 (PWM 1 kHz) sur PA0 | 15 min | Capture CubeMX |
| 2 | Fade in / fade out sur la LED | 15 min | Démo |
| 3 | Mesurer la fréquence à l'oscilloscope sur PA0 | 10 min | Mesure |
| 4 | Configurer TIM2_CH2 (50 Hz) sur PA1 | 10 min | Capture |
| 5 | Balayage 0° → 180° → 0° sur le servo | 20 min | Démo |
| 6 | Contrôler le servo par bouton (0° / 90° / 180°) | 15 min | Code |
| 7 | Rédiger le compte-rendu | 5 min | CR |

### ⚙️ Code — LED fade + Servo balayage

In [ ]:
/* ============================================================
   TP6 - LED dimming + servo moteur
   - PA0 : TIM2_CH1 — PWM 1 kHz (LED)
   - PA1 : TIM2_CH2 — PWM 50 Hz (servo)
   ============================================================ */

#include "main.h"

TIM_HandleTypeDef htim2;

/* --- Position servo : 0..180° → T_on = 1..2 ms --- */
static uint16_t angle_vers_ccr(uint16_t angle)
{
    // T = 20 ms, ARR = 19999
    // T_on = 1000 µs + angle/180 × 1000 µs
    // CCR = (T_on / 20000) × 20000 = T_on (en µs)
    return 1000 + (uint32_t)angle * 1000 / 180;
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_TIM2_Init();

    HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_1);   // LED
    HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_2);   // servo

    while (1)
    {
        /* --- 1) Fade in/out sur LED (TIM2_CH1) --- */
        for (uint16_t ccr = 0; ccr <= 1000; ccr += 5)
        {
            __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_1, ccr);
            HAL_Delay(5);
        }
        for (uint16_t ccr = 1000; ccr > 0; ccr -= 5)
        {
            __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_1, ccr);
            HAL_Delay(5);
        }

        /* --- 2) Balayage servo (TIM2_CH2) --- */
        for (uint16_t a = 0; a <= 180; a += 5)
        {
            __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_2, angle_vers_ccr(a));
            HAL_Delay(20);
        }
        for (uint16_t a = 180; a > 0; a -= 5)
        {
            __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_2, angle_vers_ccr(a));
            HAL_Delay(20);
        }
    }
}

### 🔍 Analyse du code

| Élément | Rôle |
|---|---|
| `HAL_TIM_PWM_Start()` | Démarre un canal PWM |
| `__HAL_TIM_SET_COMPARE()` | Modifie le duty en temps réel |
| `angle_vers_ccr()` | Convertit un angle en valeur CCR |
| `HAL_Delay(20)` | Laisse le temps au servo de bouger |

### 📖 Configuration CubeMX

**TIM2_CH1 (LED) :**
- Mode : PWM Generation CH1
- PSC = 71, ARR = 999 → **F = 1 kHz**
  ```
  F = 72e6 / (72 × 1000) = 1000 Hz
  ```
- Pulse (CCR) initial = 0

**TIM2_CH2 (servo) :**
- Mode : PWM Generation CH2
- PSC = 71, ARR = 19999 → **F = 50 Hz**
  ```
  F = 72e6 / (72 × 20000) = 50 Hz
  ```
- Pulse initial = 1500 (position 90°)

### 🐍 Simulation Python — Balayage servo (15 min)

Simulons la commande d'un servo et calculons les paramètres optimaux.

In [ ]:
# ============================================================
# Simulateur de commande servo
# ============================================================

def ccr_pour_angle(angle, arr=19999):
    """Retourne CCR pour une position angulaire 0..180°."""
    # T = 20 ms, T_on entre 1 et 2 ms
    t_on_ms = 1.0 + (angle / 180.0) * 1.0
    duty = t_on_ms / 20.0
    return int(duty * (arr + 1))

def afficher_balayage(angles, largeur=40):
    """Affiche la position du servo sous forme d'ASCII."""
    for a in angles:
        pos = int(a / 180 * (largeur - 1))
        ligne = " " * pos + "▲"
        print(f"{a:>4}°  |{ligne:<{largeur}}|")

print("🎯 Balayage servo 0° → 180°\n")
afficher_balayage(range(0, 181, 15))

print("\n📊 Paramètres de configuration :")
print(f"  ARR = 19999 (T = 20 ms)")
print(f"  Pour 0°   → CCR = {ccr_pour_angle(0)}")
print(f"  Pour 90°  → CCR = {ccr_pour_angle(90)}")
print(f"  Pour 180° → CCR = {ccr_pour_angle(180)}")
print(f"\n  Résolution angulaire : ~{180/1000:.2f}° par incrément de CCR")

### 🐍 Simulation Python — Loi de perception LED (15 min)

In [ ]:
# ============================================================
# Dimming LED : loi linéaire vs loi gamma
# ============================================================

def duty_lineaire(v, vmax=255):
    return int(v / vmax * 100)

def duty_gamma(v, vmax=255, gamma=2.2):
    return int((v / vmax) ** gamma * 100)

print("💡 Comparaison linéaire vs gamma (dimming LED)\n")
print(f"{'Valeur':<8}{'Linéaire':<12}{'Gamma 2.2':<14}{'Réel perçu'}")
print("-" * 50)
for v in range(0, 256, 32):
    lin = duty_lineaire(v)
    gam = duty_gamma(v)
    barre = "█" * (gam // 5)
    print(f"{v:<8}{lin:>3}%{'':<8}{gam:>3}%{'':<9}{barre}")

print("\n📌 À retenir :")
print("  → Loi linéaire : montée brusque en luminosité perçue.")
print("  → Loi gamma : montée douce et naturelle.")
print("  → Utiliser une LUT (lookup table) ou une formule au carré.")

### 📝 Compte-rendu de TP6

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX**
- TIM2_CH1 : PSC = ... , ARR = ... → F = ... Hz
- TIM2_CH2 : PSC = ... , ARR = ... → F = ... Hz
- Broches : PA0 = ... , PA1 = ...

**2. Code ajouté dans `main.c`**
```c
// Colle ici ton code
```

**3. Mesures à l'oscilloscope**
- Fréquence mesurée PA0 (LED) : ... Hz
- Fréquence mesurée PA1 (servo) : ... Hz
- Écart avec les valeurs attendues : ...

**4. Observations LED**
- Fade in/out fluide ? ...
- À partir de quel duty la LED est-elle visible ? ...
- Différence entre loi linéaire et gamma : ...

**5. Observations servo**
- Position 0° : ...
- Position 90° : ...
- Position 180° : ...
- Stabilité / vibrations : ...

**6. Problèmes rencontrés**
- ...

**7. Solutions apportées**
- ...

### 🧪 Exercice bonus — Contrôle RGB

Contrôler une **LED RGB** (3 canaux PWM) pour générer les couleurs suivantes en boucle :
- Rouge (255, 0, 0)
- Vert (0, 255, 0)
- Bleu (0, 0, 255)
- Jaune (255, 255, 0)
- Cyan (0, 255, 255)
- Magenta (255, 0, 255)
- Blanc (255, 255, 255)

**Indication :** utiliser TIM3_CH1, CH2, CH3 sur PA6, PA7, PB0.

**Question :** comment gérer proprement un tableau de couleurs en C ?

In [ ]:
// Squelette solution bonus

typedef struct {
    uint16_t r;
    uint16_t g;
    uint16_t b;
    const char *nom;
} Couleur;

static const Couleur couleurs[] = {
    {255, 0,   0,   "Rouge"},
    {0,   255, 0,   "Vert"},
    {0,   0,   255, "Bleu"},
    {255, 255, 0,   "Jaune"},
    {0,   255, 255, "Cyan"},
    {255, 0,   255, "Magenta"},
    {255, 255, 255, "Blanc"},
};

#define NB_COULEURS (sizeof(couleurs) / sizeof(couleurs[0]))

// À compléter dans la boucle principale :
// 1. Parcourir le tableau
// 2. Convertir chaque composante 0..255 en CCR (via loi gamma)
// 3. Écrire dans les 3 canaux TIM3
// 4. Attendre 500 ms

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Calculs PWM (30 min)

Pour F_clk = **72 MHz**, calculer PSC, ARR, CCR pour chaque cas.

| # | F_pwm | Duty | PSC | ARR | CCR |
|---|---|---|---|---|---|
| 1 | 1 kHz | 50 % | ? | ? | ? |
| 2 | 1 kHz | 25 % | ? | ? | ? |
| 3 | 10 kHz | 75 % | ? | ? | ? |
| 4 | 50 Hz | 7.5 % (servo 90°) | ? | ? | ? |
| 5 | 50 Hz | 10 % (servo 180°) | ? | ? | ? |
| 6 | 20 kHz | 60 % (moteur) | ? | ? | ? |
| 7 | 2 kHz | 50 % (buzzer) | ? | ? | ? |
| 8 | 500 Hz | 0 % | ? | ? | ? |
| 9 | 500 Hz | 100 % | ? | ? | ? |
| 10 | 1 Hz | 50 % (LED clignote lentement) | ? | ? | ? |

👉 Utilise le calculateur Python ci-dessous pour vérifier.

In [ ]:
# Corrigé Exercice 1

def calc_pwm(f_clk_hz, f_pwm_hz, duty, psc_pref=72, arr_max=65535):
    total = f_clk_hz / f_pwm_hz
    if total % psc_pref != 0:
        return None
    arr = int(total / psc_pref) - 1
    if arr > arr_max:
        return None
    ccr = int(duty * (arr + 1))
    return (psc_pref - 1, arr, ccr)

cas = [
    (1000,   0.50, "1 kHz / 50%"),
    (1000,   0.25, "1 kHz / 25%"),
    (10000,  0.75, "10 kHz / 75%"),
    (50,     0.075, "50 Hz / 7.5% (servo 90°)"),
    (50,     0.10, "50 Hz / 10% (servo 180°)"),
    (20000,  0.60, "20 kHz / 60% (moteur)"),
    (2000,   0.50, "2 kHz / 50% (buzzer)"),
    (500,    0.00, "500 Hz / 0%"),
    (500,    1.00, "500 Hz / 100%"),
    (1,      0.50, "1 Hz / 50% (LED lent)"),
]

F_CLK = 72_000_000

print(f"{'Cas':<28}{'PSC':<8}{'ARR':<10}{'CCR':<10}{'Vérif'}")
print("-" * 70)
for f_pwm, duty, nom in cas:
    sol = calc_pwm(F_CLK, f_pwm, duty)
    if sol:
        psc, arr, ccr = sol
        f_reelle = F_CLK / ((psc + 1) * (arr + 1))
        d_reel  = ccr / (arr + 1)
        verif = f"{f_reelle:.2f} Hz, {d_reel*100:.2f}%"
        print(f"{nom:<28}{psc:<8}{arr:<10}{ccr:<10}{verif}")
    else:
        print(f"{nom:<28}❌ impossible avec PSC+1=72")

### 🧩 Exercice 2 — Analyse d'un signal PWM (30 min)

On observe le signal PWM suivant à l'oscilloscope :

```
    ┌────┐          ┌────┐          ┌────┐
    │    │          │    │          │    │
────┘    └──────────┘    └──────────┘    └────
    ◄────────────── 4 ms ─────────────────►
```

**Questions :**
1. Quelle est la période T ?
2. Quelle est la fréquence F ?
3. Quel est le temps haut T_on (mesuré à 1 ms) ?
4. Quel est le duty cycle ?
5. Quel est le CCR si ARR = 999 ?
6. Quel est le PSC si F_clk = 72 MHz ?

In [ ]:
# Corrigé Exercice 2

F_CLK   = 72_000_000
T_ms    = 4        # période observée
T_on_ms = 1        # temps haut

F_pwm   = 1000 / T_ms
duty    = T_on_ms / T_ms
ARR     = 999      # choisi
CCR     = int(duty * (ARR + 1))
total   = F_CLK / F_pwm       # = (PSC+1)*(ARR+1)
PSC     = int(total / (ARR + 1)) - 1

print(f"T         = {T_ms} ms")
print(f"F         = {F_pwm:.2f} Hz")
print(f"T_on      = {T_on_ms} ms")
print(f"Duty      = {duty*100:.1f} %")
print(f"ARR       = {ARR}")
print(f"CCR       = {CCR}")
print(f"PSC       = {PSC}")
print()
print("Vérification :")
print(f"  F_pwm = 72e6 / ({(PSC+1)} × {(ARR+1)}) = {F_CLK/((PSC+1)*(ARR+1)):.2f} Hz")

### 🧩 Exercice 3 — Lecture du RM0008 (30 min)

Lire la section **PWM** du chapitre 15 (TIM) du RM0008 et répondre :

1. Quelle est la différence entre **PWM mode 1** et **PWM mode 2** ?
2. Que signifie **OC1REF** et comment est-il relié à la sortie physique ?
3. Que fait le bit **OC1PE** (Output Compare Preload Enable) ?
4. Pourquoi utiliser **preload** sur ARR (ARPE) ?
5. Que se passe-t-il si on écrit CCR1 > ARR ?
6. Quelle est la différence entre **edge-aligned** et **center-aligned** en termes de spectre ?

### ✍️ Réponses — Exercice 3

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

---

## 🧮 Exercice supplémentaire — Filtre passe-bas RC (optionnel)

Un signal PWM de 1 kHz filtré par un RC (R = 1 kΩ, C = 100 µF) produit une tension continue.

**Calculer :**
- La fréquence de coupure du filtre
- L'atténuation du résidu PWM à 1 kHz
- La tension moyenne de sortie si Duty = 50 % et V_high = 3.3 V

In [ ]:
# ============================================================
# Filtrage RC d'un signal PWM
# ============================================================

import math

def analyse_filtre_rc(R_ohm, C_farad, f_pwm_hz, V_high=3.3, duty=0.5):
    fc = 1 / (2 * math.pi * R_ohm * C_farad)
    att = 20 * math.log10(f_pwm_hz / fc)   # en dB
    V_moy = duty * V_high
    V_residus = V_high / (10 ** (att / 20)) / math.pi  # amplitude approx.
    return {
        "F_coupure (Hz)": fc,
        "Atténuation (dB)": att,
        "V_moy (V)": V_moy,
        "V_résidus (mV)": V_residus * 1000,
    }

print("📉 Analyse filtre RC pour PWM\n")
for R, C, f, d in [(1000, 100e-6, 1000, 0.5),
                   (1000, 10e-6, 1000, 0.5),
                   (4700, 100e-6, 1000, 0.25),
                   (1000, 100e-6, 10000, 0.5)]:
    r = analyse_filtre_rc(R, C, f, duty=d)
    print(f"R = {R:>5} Ω, C = {C*1e6:>5.0f} µF, F_pwm = {f:>6} Hz, Duty = {d*100:>4.0f} %")
    for k, v in r.items():
        print(f"    {k:<20} = {v:.3f}")
    print()

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 6

Coche ce que tu maîtrises.

- [ ] Je sais définir le PWM (période, fréquence, duty).
- [ ] Je connais la formule F_pwm = F_clk / ((PSC+1)×(ARR+1)).
- [ ] Je sais calculer CCR pour un duty donné.
- [ ] Je connais la différence entre PWM mode 1 et mode 2.
- [ ] Je sais choisir la fréquence selon l'application.
- [ ] Je connais les broches PWM du STM32F103C6T6.
- [ ] Je sais configurer un canal PWM en CubeMX.
- [ ] Je sais utiliser `HAL_TIM_PWM_Start()`.
- [ ] Je sais modifier le duty avec `__HAL_TIM_SET_COMPARE()`.
- [ ] J'ai implémenté un fade in/out sur LED.
- [ ] J'ai piloté un servo moteur.
- [ ] J'ai mesuré le signal PWM à l'oscilloscope.
- [ ] J'ai rédigé mon compte-rendu de TP6.
- [ ] J'ai lu la section PWM du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour le QCM S7 |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 6 |

---
# 📚 RESSOURCES Semaine 6

### Documents officiels
- 📄 **RM0008** — chapitre 15 (TIM) — section « PWM mode »
- 📄 **Datasheet STM32F103x6** — section 2.3.11 (Timers)
- 📄 **UM1850** — HAL TIM PWM documentation

### Outils
- **STM32CubeMX** — Timers → Channel → PWM Generation
- **Oscilloscope** ou **analyseur logique** pour mesurer F et duty
- **Servo moteur SG90** (ou équivalent) pour tester

### Vidéos
- *STM32 PWM Tutorial (HAL)* — ControllersTech
- *STM32 Servo Motor Control* — YouTube

### Bonnes pratiques
- Choisir F_pwm > 1 kHz pour LED (éviter le scintillement).
- Choisir F_pwm > 20 kHz pour moteur (éviter le sifflement).
- Utiliser 50 Hz pour un servo RC.
- Utiliser une loi gamma pour le dimming LED.
- Ne pas dépasser les capacités de la broche (20 mA max).

---

### 🔗 Passage à la semaine 7

**Prochaine séance :** Évaluation théorique — **QCM 40 questions**  
- Architecture ARM Cortex-M3
- Hardware STM32F103C6T6
- GPIO / EXTI / TIMER / PWM
- Calculs (PSC, ARR, CCR)

**Préparation :** Réviser les semaines 1 à 6 — relire les notebooks, refaire les QCM formatifs.

---

**Fin du notebook — Semaine 6** ✨